In [1]:
!pip install vllm==0.10.1


In [2]:
!pip install "mcp[all]" --upgrade


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.5/175.5 kB 11.4 MB/s eta 0:00:00
  Attempting uninstall: mcp
    Found existing installation: mcp 1.21.2
    Uninstalling mcp-1.21.2:
      Successfully uninstalled mcp-1.21.2


In [3]:
!nohup vllm serve Qwen/Qwen3-1.7B \
  --enable-auto-tool-choice \
  --tool-call-parser hermes \
  --reasoning-parser deepseek_r1 \
  --gpu-memory-utilization 0.6 \
  --max-model-len 2048 \
  > vllm.log &

nohup: redirecting stderr to stdout


In [4]:
!tail -n 20 vllm.log

(APIServer pid=2517) INFO 11-21 22:00:09 [launcher.py:37] Route: /v1/responses/{response_id}/cancel, Methods: POST
(APIServer pid=2517) INFO 11-21 22:00:09 [launcher.py:37] Route: /v1/chat/completions, Methods: POST
(APIServer pid=2517) INFO 11-21 22:00:09 [launcher.py:37] Route: /v1/completions, Methods: POST
(APIServer pid=2517) INFO 11-21 22:00:09 [launcher.py:37] Route: /v1/embeddings, Methods: POST
(APIServer pid=2517) INFO 11-21 22:00:09 [launcher.py:37] Route: /pooling, Methods: POST
(APIServer pid=2517) INFO 11-21 22:00:09 [launcher.py:37] Route: /classify, Methods: POST
(APIServer pid=2517) INFO 11-21 22:00:09 [launcher.py:37] Route: /score, Methods: POST
(APIServer pid=2517) INFO 11-21 22:00:09 [launcher.py:37] Route: /v1/score, Methods: POST
(APIServer pid=2517) INFO 11-21 22:00:09 [launcher.py:37] Route: /v1/audio/transcriptions, Methods: POST
(APIServer pid=2517) INFO 11-21 22:00:09 [launcher.py:37] Route: /v1/audio/translations, Methods: POST
(APIServer pid=2517) INFO 11-

In [5]:
!pip install openai

In [6]:
server_code = """
import sys
import asyncio
from mcp.server.fastmcp import FastMCP

# Inizializza il server
mcp = FastMCP("weather-tools")

@mcp.tool()
async def get_current_temperature(location: str, unit: str = "celsius"):
    return {
        "temperature": 26.1,
        "location": location,
        "unit": unit,
        "status": "success"
    }

@mcp.tool()
async def get_temperature_date(location: str, date: str, unit: str = "celsius"):
    return {
        "temperature": 25.9,
        "location": location,
        "date": date,
        "unit": unit
    }

if __name__ == "__main__":
    # Avvia il server su stdio
    try:
        mcp.run(transport='stdio')
    except Exception as e:
        # Scrive l'errore su stderr così possiamo leggerlo nel notebook se fallisce
        sys.stderr.write(f"Errore Server: {e}\\n")
"""

with open("server.py", "w") as f:
    f.write(server_code)

print("✅ File server.py ricreato e pulito!")

✅ File server.py ricreato e pulito!


### Ciclo di Comunicazione MCP su STDIO

| FASE | MITTENTE | CANALE | DESTINATARIO | DESCRIZIONE |
| :--- | :--- | :--- | :--- | :--- |
| **1. Richiesta RPC** | Il tuo script (Client) | `server_process.stdin` | Server MCP | Il Client costruisce il messaggio **JSON-RPC** e lo **scrive** nel flusso di ingresso del server. |
| **2. Esecuzione Tool** | Server MCP | Interno | Server MCP | Il Server legge la richiesta, la decodifica, **esegue la funzione Python** (es. `get_current_temperature`), e ottiene il risultato grezzo. |
| **3. Risposta RPC** | Server MCP | `server_process.stdout` | Il tuo script (Client) | Il Server incapsula il risultato grezzo in un messaggio **JSON-RPC di risposta** e lo **scrive** nel suo flusso di uscita. |
| **4. Formattazione Finale** | Il tuo script (Client) | Interno | LLM | Il Client legge la risposta JSON-RPC, ne estrae il contenuto (la temperatura, etc.), e lo utilizza per costruire il messaggio finale da inviare al modello vLLM. |

In [29]:
import subprocess #subprocess permette di avviare server.py e di interagire con i suoi stdin/stdout/stderr
import json
import sys
import time
from openai import OpenAI

# --- CONFIGURAZIONE ---
VLLM_API_URL = "http://localhost:8000/v1"
MCP_SERVER_SCRIPT = "server.py"

'''
metodo manuale per collegare server.py e client attraverso subprocess:
- avvia server.py
- Scrive manualmente la stringa JSON-RPC(formato specifico con cui i messaggi MCP sono strutturati.)
  in server_process.stdin
- Legge la stringa JSON-RPC di risposta di server_process.stdout

Unico modo per evitare error: fileno -> dovrebbe essere un problema specifico di colab quando si usando stdin/stdout
'''


# 1. AVVIO DEL SERVER MCP
print(" Avvio del Server MCP...")
server_process = subprocess.Popen(
    ['python', MCP_SERVER_SCRIPT],
    stdin=subprocess.PIPE, # standard Iput: canale utilizzato per ricevere dati in input
    stdout=subprocess.PIPE, # standard output: canale utilizzato per il visualizzare output
    stderr=subprocess.PIPE, # standard error: utilizzato per visualizzare errori
    text=True
)

# --- FUNZIONE DI INIZIALIZZAZIONE MCP  ---
def initialize_mcp():
    """Esegue l'inizializzazione manuale del protocollo MCP."""
    print("Eseguo inizializzaione manuale MCP...")
    # 1. Invia Initialize
    init_request = {
        "jsonrpc": "2.0",
        "method": "initialize",
        "params": {
            "protocolVersion": "2024-11-05",
            "capabilities": {},
            "clientInfo": {"name": "my-notebook", "version": "1.0"}
        },
        "id": 0
    }
    server_process.stdin.write(json.dumps(init_request) + "\n")
    server_process.stdin.flush()

    # Leggi risposta initialize
    response = server_process.stdout.readline()
    if not response:
        print("Errore: Il server non ha risposto all'initialize")
        return False

    # 2. Invia Notifica "Initialized" (Conferma)
    notify_request = {
        "jsonrpc": "2.0",
        "method": "notifications/initialized",
        "params": {}
    }
    server_process.stdin.write(json.dumps(notify_request) + "\n")
    server_process.stdin.flush()
    print("Server MCP Inizializzato e pronto.")
    return True

# Avvia e inizializza subito
time.sleep(1)
if not initialize_mcp():
    print(" Impossibile inizializzare il server. Terminazione.")
    server_process.terminate()
    sys.exit(1)

# 2. CLIENT vLLM
client = OpenAI(base_url=VLLM_API_URL, api_key="EMPTY")

tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_current_temperature",
            "description": "Get current temperature at a location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string", "description": "City and State"},
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}
                },
                "required": ["location", "unit"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_temperature_date",
            "description": "Get temperature for a specific date",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string"},
                    "date": {"type": "string"},
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}
                },
                "required": ["location", "date", "unit"]
            }
        }
    }
]


def call_mcp_tool(tool_name, tool_args):
  '''
  Translate an LLM-style tool call into an MCP JSON-RPC request and send it over STDIN to the MCP server.
  '''
  print(f"MCP Call: {tool_name} {tool_args}")

  # COSTRUZIONE DELLA RICHIESTA SECONDO STANDARD MCP
  rpc_request = {
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {
            "name": tool_name,
            "arguments": tool_args
        },
        "id": int(time.time())
    }

  try:
        server_process.stdin.write(json.dumps(rpc_request) + "\n")
        server_process.stdin.flush()
        response_line = server_process.stdout.readline()
  except BrokenPipeError:
        return {"error": "Server Crashed"}

  if not response_line: return {"error": "Nessuna risposta"}

  try:
        data = json.loads(response_line)
        # MCP restituisce il risultato annidato in "result" -> "content"
        if "error" in data:
            return f"MCP Error: {data['error']['message']}"

        # Estrazione risultato standard MCP
        if "result" in data:
             # A volte il risultato è un testo, a volte un oggetto "content"
             return data["result"].get("content", data["result"])

        return data
  except: return {"error": "JSON invalido"}

# Main flow: ask vLLM, detect tool calls, call MCP, return final answer
try:
    user_query = "What's the temperature in San Francisco now? And tomorrow (2024-10-01)?"
    print(f" Utente: {user_query}")
    messages = [{"role": "user", "content": user_query}]

    print(" scarico qwen 3-1.7B da vLLM...")
    response = client.chat.completions.create(
        model="Qwen/Qwen3-1.7B",
        messages=messages,
        tools=tools_schema,
        tool_choice="auto"
    )

    tool_calls = response.choices[0].message.tool_calls

    if tool_calls:
        print("qwen chiede un tool")
        messages.append(response.choices[0].message)
        for tool_call in tool_calls:
            fname = tool_call.function.name
            fargs = json.loads(tool_call.function.arguments)

            # Chiamata corretta
            res = call_mcp_tool(fname, fargs)
            print(f" Risultato: {res}")

            messages.append({
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": fname,
                "content": json.dumps(res) if not isinstance(res, str) else res
            })

        final = client.chat.completions.create(
            model="Qwen/Qwen3-1.7B", messages=messages
        )
        print("\n RISPOSTA FINALE:\n", final.choices[0].message.content)
    else:
        print("qwen non ha usato tool.")

finally:
    server_process.terminate()

 Avvio del Server MCP...
Eseguo inizializzaione manuale MCP...
Server MCP Inizializzato e pronto.
 Utente: What's the temperature in San Francisco now? And tomorrow (2024-10-01)?
 scarico qwen 3-1.7B da vLLM...
qwen chiede un tool
MCP Call: get_current_temperature {'location': 'San Francisco', 'unit': 'celsius'}
 Risultato: [{'type': 'text', 'text': '{\n  "temperature": 26.1,\n  "location": "San Francisco",\n  "unit": "celsius",\n  "status": "success"\n}'}]
MCP Call: get_temperature_date {'location': 'San Francisco', 'date': '2024-10-01', 'unit': 'celsius'}
 Risultato: [{'type': 'text', 'text': '{\n  "temperature": 25.9,\n  "location": "San Francisco",\n  "date": "2024-10-01",\n  "unit": "celsius"\n}'}]

 RISPOSTA FINALE:
 

The current temperature in San Francisco is **26.1°C**.  

For **tomorrow (2024-10-01)**, the temperature is **25.9°C**.  

Note: The second response appears to reference the same date as the current query, which may be an inconsistency. However, based on the provi